In [14]:
import pandas as pd
from pyfamsa import Aligner, Sequence
from Bio import SeqIO
from stats import align_pair_famsa, compute_stats
from AlignmentViewer import AlignmentViewer

In [5]:
new_seqs = pd.read_excel("../../data/new-guinea-data/GIN-LASV-final-ViMOP-seq-for-Leuven_V02.xlsx")

In [6]:
tchain = ["G0780","G0795","G0796","G0797"]
new_seqs_tchain = new_seqs[new_seqs["seq-ID"].isin(tchain)]

In [8]:
new_seqs_tchain.columns

Index(['Nr', 'seq-ID', 'Date-sampling', 'latitude', 'longtitute', 'precision',
       'Coverage S [%|', 'Coverage L  [%|', 'Reference S', 'Sequence S',
       'Reference L', 'Sequence L', 'Genebank ID S', 'Genebank ID L'],
      dtype='str')

In [10]:

def resolve_col(df, wanted):
    # case-insensitive column resolver
    m = {c.lower(): c for c in df.columns}
    if wanted.lower() not in m:
        raise KeyError(f"Column '{wanted}' not found. Available: {list(df.columns)}")
    return m[wanted.lower()]

def pairwise_stats_from_df(df, seq_col, id_col="seq-ID", do_align=True):
    seq_col = resolve_col(df, seq_col)
    id_col = resolve_col(df, id_col)

    # keep only rows with both ID and sequence
    work = df[[id_col, seq_col]].dropna().copy()
    work[id_col] = work[id_col].astype(str)
    work[seq_col] = work[seq_col].astype(str)

    rows = []
    for (id1, s1), (id2, s2) in itertools.combinations(work[[id_col, seq_col]].itertuples(index=False, name=None), 2):
        seq1 = s1.upper()
        seq2 = s2.upper()

        if do_align:
            # pyFAMSA pairwise alignment from stats.py
            a1, a2 = align_pair_famsa(id1, seq1.replace("-", ""), id2, seq2.replace("-", ""))
            st = compute_stats(a1, a2)
        else:
            # requires equal lengths if already aligned
            st = compute_stats(seq1, seq2)

        rows.append({
            "segment": seq_col,
            "seq1_id": id1,
            "seq2_id": id2,
            **st,
        })

    return pd.DataFrame(rows)

# Apply to your tchain dataframe
L_pairwise = pairwise_stats_from_df(new_seqs_tchain, "Sequence L", id_col="seq-ID", do_align=True)
S_pairwise = pairwise_stats_from_df(new_seqs_tchain, "Sequence S", id_col="seq-ID", do_align=True)

pairwise_all = pd.concat([L_pairwise, S_pairwise], ignore_index=True)

print("L comparisons:", len(L_pairwise))
display(L_pairwise)

print("S comparisons:", len(S_pairwise))
display(S_pairwise)

# Optional save
pairwise_all.to_csv("../../data/new-guinea-data/tchain_pairwise_LS_stats.tsv", sep="\t", index=False)

L comparisons: 6


,segment,seq1_id,seq2_id,alignment_length,comparable_sites,matches,mismatches,mismatch_rate,seq1_length,seq2_length,seq1_gaps,seq2_gaps,seq1_Ns,seq2_Ns
0,Sequence L,G0780,G0795,7249,6993,2878,4,0.000572,7248,6994,1,255,4183,174
1,Sequence L,G0780,G0796,7249,7245,3059,3,0.000414,7248,7246,1,3,4183,18
2,Sequence L,G0780,G0797,7249,7235,3060,4,0.000553,7248,7236,1,13,4183,31
3,Sequence L,G0795,G0796,7246,6994,6815,1,0.000143,6994,7246,252,0,174,18
4,Sequence L,G0795,G0797,7236,6994,6806,1,0.000143,6994,7236,242,0,174,31
5,Sequence L,G0796,G0797,7246,7236,7198,2,0.000276,7246,7236,0,10,18,31


S comparisons: 6


,segment,seq1_id,seq2_id,alignment_length,comparable_sites,matches,mismatches,mismatch_rate,seq1_length,seq2_length,seq1_gaps,seq2_gaps,seq1_Ns,seq2_Ns
0,Sequence S,G0780,G0795,3515,3179,539,653,0.205411,3402,3292,113,223,2189,42
1,Sequence S,G0780,G0796,3590,3191,572,622,0.194923,3402,3379,188,211,2189,56
2,Sequence S,G0780,G0797,3515,3186,539,653,0.204959,3402,3299,113,216,2189,47
3,Sequence S,G0795,G0796,3379,3292,3243,1,0.000304,3292,3379,87,0,42,56
4,Sequence S,G0795,G0797,3299,3292,3245,0,0.000000,3292,3299,7,0,42,47
5,Sequence S,G0796,G0797,3379,3299,3248,1,0.000303,3379,3299,0,80,56,47


In [15]:
from pathlib import Path
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

# Export the 4 selected S sequences from the full pre-aligned S alignment
export_s_fasta = Path("../../data/new-guinea-data/tchain_from_full_alignment_S.fasta")

selected_s = []
for header, seq in s_records:
    matched = next((n for n in names if n in header), None)
    if matched is not None:
        selected_s.append((matched, seq))

if len(selected_s) < 2:
    raise ValueError(f"Expected >=2 selected S sequences, found {len(selected_s)}")

# De-duplicate by ID if any repeated header matches exist (keep first)
seen = set()
selected_s_unique = []
for sid, sseq in selected_s:
    if sid not in seen:
        selected_s_unique.append((sid, sseq))
        seen.add(sid)

records = [SeqRecord(Seq(seq), id=sid, description="") for sid, seq in selected_s_unique]
SeqIO.write(records, export_s_fasta, "fasta")


sequences = list(SeqIO.parse(str(export_s_fasta), "fasta"))
AlignmentViewer.display_alignment(sequences, nseqs=5, as_html=True, show_consensus=True, color_snps_only=True)

In [11]:
from pathlib import Path
import pandas as pd

from stats import read_fasta, pairwise_stats, tsv_header, write_tsv

# Names to extract from full alignments
names = new_seqs_tchain["seq-ID"].dropna().astype(str).tolist()
print("Names used:", names)

# Already-aligned full alignment files (no re-alignment)
s_aln = Path("../../data/alignments/final/LASV_GENBANK_20250725_curated.added_background_S.renamed.msa")
l_aln = Path("../../data/alignments/final/LASV_GENBANK_20250725_curated.added_background_L.renamed.msa")

# Read alignments
s_records = read_fasta(s_aln)
l_records = read_fasta(l_aln)
print(f"S alignment records: {len(s_records)}")
print(f"L alignment records: {len(l_records)}")

# Pairwise stats on extracted sequences (upper triangle only)
s_rows = pairwise_stats(s_records, names, unaligned=False)
l_rows = pairwise_stats(l_records, names, unaligned=False)

s_df = pd.DataFrame(s_rows, columns=tsv_header())
l_df = pd.DataFrame(l_rows, columns=tsv_header())

print("S comparisons:", len(s_df))
display(s_df)

print("L comparisons:", len(l_df))
display(l_df)

# Save
s_out = Path("../../data/new-guinea-data/tchain_pairwise_from_full_alignment_S.tsv")
l_out = Path("../../data/new-guinea-data/tchain_pairwise_from_full_alignment_L.tsv")
write_tsv(s_out, tsv_header(), s_rows)
write_tsv(l_out, tsv_header(), l_rows)

print("Saved:")
print(s_out)
print(l_out)

Names used: ['G0780', 'G0795', 'G0796', 'G0797']
S alignment records: 1043
L alignment records: 521
S comparisons: 6


,seq1_id,seq2_id,alignment_length,comparable_sites,matches,mismatches,mismatch_rate,seq1_length,seq2_length,seq1_gaps,seq2_gaps,seq1_Ns,seq2_Ns
0,G0780,G0795,3186,3186,1212,1,0.000314,3186,3186,0,0,1973,1
1,G0780,G0796,3186,3186,1212,1,0.000314,3186,3186,0,0,1973,4
2,G0780,G0797,3186,3186,1212,1,0.000314,3186,3186,0,0,1973,4
3,G0795,G0796,3186,3186,3181,1,0.000314,3186,3186,0,0,1,4
4,G0795,G0797,3186,3186,3182,0,0.000000,3186,3186,0,0,1,4
5,G0796,G0797,3186,3186,3180,1,0.000314,3186,3186,0,0,4,4


L comparisons: 6


,seq1_id,seq2_id,alignment_length,comparable_sites,matches,mismatches,mismatch_rate,seq1_length,seq2_length,seq1_gaps,seq2_gaps,seq1_Ns,seq2_Ns
0,G0780,G0795,7017,6886,2870,4,0.000581,6962,6887,55,130,3999,157
1,G0780,G0796,7017,6962,2957,3,0.000431,6962,6963,55,54,3999,6
2,G0780,G0797,7017,6962,2958,4,0.000575,6962,6963,55,54,3999,9
3,G0795,G0796,7017,6887,6725,1,0.000145,6887,6963,130,54,157,6
4,G0795,G0797,7017,6887,6721,1,0.000145,6887,6963,130,54,157,9
5,G0796,G0797,7017,6963,6947,2,0.000287,6963,6963,54,54,6,9


Saved:
../../data/new-guinea-data/tchain_pairwise_from_full_alignment_S.tsv
../../data/new-guinea-data/tchain_pairwise_from_full_alignment_L.tsv
